# Phase D: Phasor analysis

Steps:
9.  Load position annotations and merge into sdt_df; report coverage
10. Per-image intensity mask (Otsu threshold on photon count; percentile fallback)
11. Demo: calibrated phasor for one file per (fixation_type, em_filter_nm)
12. Full per-file phasor summary -> results/sdt_phasor_summary.csv
13. Hexbin phasor plots on universal semicircle, grouped by annotation x cell_type

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from sdtfile import SdtFile

# -- Configuration ----------------------------------------------------------
WIN_DATA_DIRS = [
    Path(r"E:\18_RK_Circadian\data\raw\20260429_KPC_fixed_dishes_on_SLIM"),
    Path(r"E:\18_RK_Circadian\data\raw\20260501_KPC_fixed_dishes_on_SLIM"),
    Path(r"E:\18_RK_Circadian\data\raw\20260509_KPC_fixed_dishes_on_SLIM"),
    Path(r"E:\18_RK_Circadian\data\raw\20260508_KPC_live_on_SLIM"),
    Path(r"E:\18_RK_Circadian\data\raw\20260517_KPC_live_on_SLIM"),
    Path(r"E:\18_RK_Circadian\data\raw\20260520_KPC_fixed_dishes_SLIM"),
    Path(r"E:\18_RK_Circadian\data\raw\20260521_KPC_live_SLIM"),
    Path(r"E:\18_RK_Circadian\data\raw\20260522_KPC_fixed_dishes_SLIM"),
]
LIN_DATA_DIRS = [
    Path("/media/mint/BRPresbkup/18_RK_Circadian/data/raw/20260429_KPC_fixed_dishes_on_SLIM"),
    Path("/media/mint/BRPresbkup/18_RK_Circadian/data/raw/20260501_KPC_fixed_dishes_on_SLIM"),
    Path("/media/mint/BRPresbkup/18_RK_Circadian/data/raw/20260509_KPC_fixed_dishes_on_SLIM"),
    Path("/media/mint/BRPresbkup/18_RK_Circadian/data/raw/20260508_KPC_live_on_SLIM"),
    Path("/media/mint/BRPresbkup/18_RK_Circadian/data/raw/20260517_KPC_live_on_SLIM"),
    Path("/media/mint/BRPresbkup/18_RK_Circadian/data/raw/20260520_KPC_fixed_dishes_SLIM"),
    Path("/media/mint/BRPresbkup/18_RK_Circadian/data/raw/20260521_KPC_live_SLIM"),
    Path("/media/mint/BRPresbkup/18_RK_Circadian/data/raw/20260522_KPC_fixed_dishes_SLIM"),
]
CURRENT_OS = "Win"
data_dirs = WIN_DATA_DIRS if CURRENT_OS == "Win" else LIN_DATA_DIRS

REP_RATE_HZ = 80e6                    # laser repetition rate (Hz)
OMEGA       = 2.0 * np.pi * REP_RATE_HZ   # angular frequency (rad/s)

# Intensity mask: minimum fraction kept when Otsu gives degenerate result
MASK_FALLBACK_FRAC = 0.50    # keep top 50% of pixels by photon count

# Gaussian blur sigma applied to photon image before Otsu thresholding.
# Smooths shot noise so the threshold follows cell-level structure.
# sigma = FWHM / 2.355; 5-pixel FWHM -> sigma ~ 2.1
MASK_BLUR_SIGMA = 2.0

# Gaussian blur sigma for phasor smoothing (photon-weighted, spatial domain).
# Applied to G and S maps before plotting; does not affect per-file summary stats.
PHASOR_BLUR_SIGMA = 2.0

# Emission filters to include in phasor analysis
PHASOR_EM_FILTERS = [457, 535]

# Column order for cell_type in plots (left to right).
# Any cell types not listed here are appended in sorted order after.
CELL_TYPE_ORDER = ["KPCWT", "BKO"]

# Annotation groups to merge before plotting (rhs label is what appears in plots)
ANNOTATION_REMAP = {
    "colony_island": "colony_edge+island",
    "colony_edge":   "colony_edge+island",
}
# Annotation groups to exclude from plots entirely
ANNOTATION_EXCLUDE = {"no_cells", "(unannotated)"}

# If True, only files whose Phase C shift_nonzero_px == 0 (shift-zero fit) are
# included in the phasor analysis and written to sdt_phasor_summary.csv.
# Requires results/fit_qc_summary.csv (written by Phase C).
# Set False to include all files regardless of IRF shift mode.
REQUIRE_SHIFT_ZERO = True

In [ ]:
# -- Load Phase A/B outputs -------------------------------------------------
results_dir = Path("../results")

sdt_df = pd.read_csv(results_dir / "sdt_metadata_cal.csv")
fp_map = pd.read_csv(results_dir / "filepath_map.csv")
sdt_df = sdt_df.merge(fp_map[["filename", "filepath"]], on="filename", how="left")
sdt_df["filepath"]         = sdt_df["filepath"].map(Path)
sdt_df["acquisition_time"] = pd.to_datetime(sdt_df["acquisition_time"])

print(f"Loaded {len(sdt_df)} SDT files")
n_cal = sdt_df["phasor_cal_phase_rad"].notna().sum()
print(f"Files with phasor calibration: {n_cal} / {len(sdt_df)}")

## Step 9: Load position annotations

Fill results/position_annotation.csv (generated by Phase C Step 12)
before running this cell. Files with blank annotation are retained but
labelled "(unannotated)" in plots.

In [ ]:
annot_path = results_dir / "position_annotation.csv"

if annot_path.exists():
    annot_df = pd.read_csv(annot_path)
    sdt_df = sdt_df.merge(
        annot_df[["filename", "annotation", "notes"]].rename(
            columns={"annotation": "position_annotation",
                     "notes":      "position_notes"}
        ),
        on="filename", how="left",
    )
    filled = (sdt_df["position_annotation"].notna() &
              (sdt_df["position_annotation"] != ""))
    print(f"Annotations loaded: {filled.sum()} / {len(sdt_df)} files annotated")
    if filled.any():
        print(sdt_df.loc[filled, "position_annotation"].value_counts().to_string())
else:
    sdt_df["position_annotation"] = np.nan
    sdt_df["position_notes"]      = ""
    print(f"WARNING: {annot_path} not found")
    print("  Run Phase C Step 12 to generate the template, fill it, then re-run.")

# Fill blanks so groupby and plot labels work cleanly
sdt_df["position_annotation"] = (
    sdt_df["position_annotation"]
    .fillna("")
    .replace("", "(unannotated)")
)

# Restrict to sample files with calibration + target emission filters
sample_df = sdt_df[
    (sdt_df["file_type"] == "sample") &
    sdt_df["em_filter_nm"].isin(PHASOR_EM_FILTERS) &
    sdt_df["phasor_cal_phase_rad"].notna()
].copy()

if REQUIRE_SHIFT_ZERO:
    _qc_path = results_dir / "fit_qc_summary.csv"
    if _qc_path.exists():
        _qc = pd.read_csv(_qc_path)[["filename", "shift_nonzero_px"]].drop_duplicates("filename")
        sample_df = sample_df.merge(_qc, on="filename", how="left")
        # Treat missing shift_nonzero_px as 0 (no Phase C data -> assume shift=0)
        sample_df["shift_nonzero_px"] = pd.to_numeric(
            sample_df["shift_nonzero_px"], errors="coerce"
        ).fillna(0)
        _sz_mask = sample_df["shift_nonzero_px"] == 0
        _n_dropped = (~_sz_mask).sum()
        if _n_dropped:
            print(f"REQUIRE_SHIFT_ZERO: dropping {_n_dropped} files with shift_nonzero_px > 0:")
            for _fn in sample_df.loc[~_sz_mask, "filename"]:
                print(f"  {_fn}")
        sample_df = sample_df[_sz_mask].copy()
        print(f"REQUIRE_SHIFT_ZERO: {len(sample_df)} files remain")
    else:
        print(f"REQUIRE_SHIFT_ZERO: fit_qc_summary.csv not found -- "
              f"run Phase C first; including all files for now")

print(f"\nSample files for phasor analysis: {len(sample_df)}")
print(sample_df.groupby(
    ["fixation_type", "em_filter_nm", "position_annotation"], dropna=False
).size().rename("n_files").reset_index().to_string(index=False))

## Step 10: Intensity mask

Otsu threshold divides background from cell pixels on the photon count image.
When Otsu is degenerate (threshold too low or too high), fall back to
keeping the top MASK_FALLBACK_FRAC fraction of pixels.

In [ ]:
def otsu_threshold(img: np.ndarray) -> float:
    """Return Otsu threshold for a 2-D photon count image."""
    flat = img[np.isfinite(img) & (img > 0)].ravel()
    if flat.size < 2:
        return 0.0
    counts, edges = np.histogram(flat, bins=256,
                                 range=(float(flat.min()), float(flat.max())))
    total = float(counts.sum())
    if total == 0:
        return 0.0
    bin_mid = 0.5 * (edges[:-1] + edges[1:])
    mu_tot  = float((counts * bin_mid).sum()) / total
    best_var, best_thr = -1.0, float(edges[0])
    w0 = sum0 = 0.0
    for i in range(len(counts)):
        w0   += counts[i]
        sum0 += counts[i] * bin_mid[i]
        if w0 == 0 or w0 == total:
            continue
        w1   = total - w0
        mu0  = sum0 / w0
        mu1  = (mu_tot * total - sum0) / w1
        var_b = (w0 / total) * (w1 / total) * (mu0 - mu1) ** 2
        if var_b > best_var:
            best_var = var_b
            best_thr = float(edges[i + 1])
    return best_thr


def intensity_mask(photon_img: np.ndarray,
                   fallback_frac: float = MASK_FALLBACK_FRAC,
                   blur_sigma: float = MASK_BLUR_SIGMA):
    """Return (bool mask, threshold).  True = keep pixel.

    Applies a Gaussian blur (sigma=blur_sigma) before Otsu so the threshold
    follows cell-level structure rather than per-pixel shot noise.
    Falls back to top-percentile if Otsu keeps <5% or >95% of pixels.
    """
    blurred = gaussian_filter(photon_img.astype(float), sigma=blur_sigma)
    thresh  = otsu_threshold(blurred)
    mask    = blurred >= thresh
    frac    = mask.sum() / float(mask.size)
    if frac < 0.05 or frac > 0.95:
        lo     = float(np.nanpercentile(
            blurred[blurred > 0], 100.0 * (1.0 - fallback_frac)
        ))
        thresh = lo
        mask   = blurred >= lo
    return mask, thresh


def smooth_phasor(G_cal: np.ndarray, S_cal: np.ndarray,
                  photons: np.ndarray,
                  sigma: float = PHASOR_BLUR_SIGMA) -> tuple:
    """Photon-weighted Gaussian smoothing of G and S in spatial domain.

    Filters G*photons and photons separately before dividing, which is
    equivalent to averaging the underlying TCSPC decays over a local
    neighbourhood.  Returns (G_sm, S_sm) with NaN where photons == 0.
    """
    ph     = np.nan_to_num(photons, nan=0.0)
    G_num  = np.nan_to_num(G_cal * photons, nan=0.0)
    S_num  = np.nan_to_num(S_cal * photons, nan=0.0)
    ph_sm  = gaussian_filter(ph,    sigma=sigma)
    G_sm   = np.where(ph_sm > 0, gaussian_filter(G_num, sigma=sigma) / ph_sm, np.nan)
    S_sm   = np.where(ph_sm > 0, gaussian_filter(S_num, sigma=sigma) / ph_sm, np.nan)
    return G_sm, S_sm


# -- Demo: show masks for one file per (fixation_type, em_filter_nm) --------
shown = set()
for _, row in sample_df.sort_values("acquisition_time").iterrows():
    gkey = (row.get("fixation_type"), row.get("em_filter_nm"))
    if gkey in shown:
        continue
    try:
        sdt_obj = SdtFile(str(row["filepath"]))
        photons = sdt_obj.data[0].astype(float).sum(axis=2)
    except Exception as e:
        print(f"Could not load {row['filename']}: {e}")
        continue
    mask, thresh = intensity_mask(photons)
    shown.add(gkey)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    vmax = float(np.percentile(photons[photons > 0], 99))
    axes[0].imshow(photons, cmap="inferno", vmin=0, vmax=vmax)
    axes[0].set_title("photons (raw)")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray_r")
    axes[1].set_title(f"mask  Otsu thr={thresh:.0f}")
    axes[1].axis("off")

    axes[2].imshow(np.where(mask, photons, np.nan), cmap="inferno",
                   vmin=0, vmax=vmax)
    axes[2].set_title(f"masked  {100*mask.sum()/mask.size:.0f}% kept")
    axes[2].axis("off")

    plt.suptitle(f"{row.get('fixation_type')} / {row.get('em_filter_nm')} nm  "
                 f"{row['filename'][:55]}", fontsize=8)
    plt.tight_layout()
    plt.show()

## Step 11: Calibrated phasor (demo)

For each SDT file, compute G and S from the TCSPC decay histogram:

  N[i,j]    = sum_t  decay[i,j,t]          (total photons per pixel)
  G_raw[i,j] = (1/N) * sum_t  decay * cos(omega * t)
  S_raw[i,j] = (1/N) * sum_t  decay * sin(omega * t)

Apply per-file calibration from Phase B (interpolated from chroma slides):

  G_cal = mod_corr * (G_raw * cos(phi) - S_raw * sin(phi))
  S_cal = mod_corr * (G_raw * sin(phi) + S_raw * cos(phi))

where phi = phasor_cal_phase_rad, mod_corr = phasor_cal_mod.

Lifetimes from the weighted mean phasor:
  tau_phi = tan(arctan2(S,G)) / (omega * 1e-9)    [ns]
  tau_mod = sqrt(1/M^2 - 1)  / (omega * 1e-9)    [ns],   M = sqrt(G^2+S^2)

In [ ]:
def _get_time_ns(sdt_obj, n_bins: int) -> np.ndarray:
    """Return bin-centre times in nanoseconds for a loaded SdtFile."""
    t = sdt_obj.times[0].astype(float) * 1e9    # s -> ns
    if t.size == n_bins + 1:                     # edges -> midpoints
        return 0.5 * (t[:-1] + t[1:])
    if t.size == n_bins:
        return t
    # Fallback: uniform bins over 12.5 ns
    dt = 12.5 / n_bins
    return np.arange(n_bins) * dt + 0.5 * dt


def compute_phasor_raw(decay: np.ndarray,
                       time_ns: np.ndarray) -> tuple:
    """Compute raw (uncalibrated) G, S and photon count per pixel.

    Parameters
    ----------
    decay   : (ny, nx, n_timebins) float array
    time_ns : (n_timebins,) bin-centre times in ns

    Returns
    -------
    G, S, photons : each (ny, nx)
    """
    photons  = decay.sum(axis=2)
    safe_n   = np.where(photons > 0, photons, 1.0)
    omega_ns = OMEGA * 1e-9                      # rad / ns
    cos_t    = np.cos(omega_ns * time_ns)        # (n_timebins,)
    sin_t    = np.sin(omega_ns * time_ns)
    G = np.tensordot(decay, cos_t, axes=[[2], [0]]) / safe_n
    S = np.tensordot(decay, sin_t, axes=[[2], [0]]) / safe_n
    G = np.where(photons > 0, G, np.nan)
    S = np.where(photons > 0, S, np.nan)
    return G, S, photons


def apply_phasor_cal(G_raw: np.ndarray, S_raw: np.ndarray,
                     phase_corr: float, mod_corr: float) -> tuple:
    """Rotate and scale raw phasor arrays."""
    c = np.cos(phase_corr)
    s = np.sin(phase_corr)
    G_cal = mod_corr * (G_raw * c - S_raw * s)
    S_cal = mod_corr * (G_raw * s + S_raw * c)
    return G_cal, S_cal


def load_phasor(row: pd.Series) -> dict | None:
    """Load an SDT file and return calibrated phasor + mask.

    Returns dict with keys:
        G_cal, S_cal, photons, mask (all (ny, nx) arrays)
    Returns None on any failure.
    """
    phase_corr = row.get("phasor_cal_phase_rad")
    mod_corr   = row.get("phasor_cal_mod")
    if pd.isna(phase_corr) or pd.isna(mod_corr):
        return None
    try:
        sdt_obj   = SdtFile(str(row["filepath"]))
        decay_raw = sdt_obj.data[0].astype(float)
    except Exception:
        return None
    time_ns    = _get_time_ns(sdt_obj, decay_raw.shape[2])
    G_raw, S_raw, photons = compute_phasor_raw(decay_raw, time_ns)
    G_cal, S_cal          = apply_phasor_cal(G_raw, S_raw,
                                             float(phase_corr), float(mod_corr))
    mask, thresh = intensity_mask(photons)
    G_sm, S_sm   = smooth_phasor(G_cal, S_cal, photons)
    return {
        "G_cal":   G_cal,    # raw calibrated phasor (used for summary stats)
        "S_cal":   S_cal,
        "G_sm":    G_sm,     # spatially smoothed phasor (used for plots)
        "S_sm":    S_sm,
        "photons": photons,
        "mask":    mask,
        "thresh":  thresh,
    }


# -- Demo: phasor plot for one file per (fixation_type, em_filter_nm) -------
def draw_semicircle(ax, lw: float = 1.2, alpha: float = 0.45,
                    lifetime_ticks=(0.3, 0.5, 1.0, 2.0, 4.0)):
    """Draw the universal FLIM semicircle with lifetime tick marks."""
    theta = np.linspace(0.0, np.pi, 300)
    ax.plot(0.5 + 0.5 * np.cos(theta), 0.5 * np.sin(theta),
            "k-", lw=lw, alpha=alpha, zorder=2)
    omega_ns = OMEGA * 1e-9
    for tau in lifetime_ticks:
        denom = 1.0 + (omega_ns * tau) ** 2
        gx    = 1.0 / denom
        sx    = (omega_ns * tau) / denom
        ax.plot(gx, sx, "ko", ms=3, alpha=alpha, zorder=3)
        ax.annotate(f"{tau}ns", (gx, sx), fontsize=6, color="k",
                    ha="center", va="bottom", alpha=alpha + 0.2)


shown = set()
for _, row in sample_df.sort_values("acquisition_time").iterrows():
    gkey = (row.get("fixation_type"), row.get("em_filter_nm"))
    if gkey in shown:
        continue
    result = load_phasor(row)
    if result is None:
        print(f"Skipping {row['filename']} (load failed or no calibration)")
        continue
    shown.add(gkey)

    G_cal, S_cal = result["G_cal"], result["S_cal"]
    G_sm,  S_sm  = result.get("G_sm", G_cal), result.get("S_sm", S_cal)
    photons, mask = result["photons"], result["mask"]
    valid = mask & np.isfinite(G_sm) & np.isfinite(S_sm)

    G_plot = G_sm[valid].ravel()    # smoothed for display
    S_plot = S_sm[valid].ravel()
    w_plot = photons[valid].ravel()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Phasor hexbin
    ax = axes[0]
    if G_plot.size > 0:
        hb = ax.hexbin(G_plot, S_plot, C=w_plot, reduce_C_function=np.sum,
                       gridsize=60, cmap="viridis", mincnt=1)
        plt.colorbar(hb, ax=ax, label="photons", fraction=0.046, pad=0.04)
    draw_semicircle(ax)
    ax.set_xlabel("G")
    ax.set_ylabel("S")
    ax.set_title(f"Phasor (calibrated)\n{row.get('fixation_type')} / "
                 f"{row.get('em_filter_nm')} nm")
    ax.set_aspect("equal", adjustable="datalim")

    # G_cal spatial map
    g_img = np.where(valid, G_cal, np.nan)
    ax2   = axes[1]
    if np.isfinite(g_img).any():
        vlo = float(np.nanpercentile(g_img[np.isfinite(g_img)], 2))
        vhi = float(np.nanpercentile(g_img[np.isfinite(g_img)], 98))
        im2 = ax2.imshow(g_img, cmap="RdBu_r", vmin=vlo, vmax=vhi)
        plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label="G_cal")
    ax2.set_title("G_cal (spatial)")
    ax2.axis("off")

    # S_cal spatial map
    s_img = np.where(valid, S_cal, np.nan)
    ax3   = axes[2]
    if np.isfinite(s_img).any():
        vlo = float(np.nanpercentile(s_img[np.isfinite(s_img)], 2))
        vhi = float(np.nanpercentile(s_img[np.isfinite(s_img)], 98))
        im3 = ax3.imshow(s_img, cmap="RdBu_r", vmin=vlo, vmax=vhi)
        plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, label="S_cal")
    ax3.set_title("S_cal (spatial)")
    ax3.axis("off")

    plt.suptitle(row["filename"][:75], fontsize=8)
    plt.tight_layout()
    plt.show()

    if G_plot.size > 0:
        G_wm = float(np.average(G_plot, weights=w_plot))
        S_wm = float(np.average(S_plot, weights=w_plot))
        print(f"  G_cal wmean={G_wm:.4f}   S_cal wmean={S_wm:.4f}"
              f"   n_px={valid.sum()}")

## Step 12: Per-file phasor summary

Process every sample file. Results are cached in `phasor_cache` (dict keyed
by filename) and written to results/sdt_phasor_summary.csv.

Columns in the summary:
  G_cal_wmean / S_cal_wmean  -- photon-count weighted mean phasor
  tau_phi_ns                 -- phase lifetime  = tan(phi) / omega
  tau_mod_ns                 -- modulation lifetime = sqrt(1/M^2 - 1) / omega
  M_wmean                    -- modulation depth of the weighted mean phasor

In [ ]:
omega_ns = OMEGA * 1e-9    # rad / ns (used for lifetime conversions)

phasor_cache   = {}    # filename -> load_phasor() result dict (full arrays)
phasor_records = []

n_total = len(sample_df)
for i, (_, row) in enumerate(sample_df.iterrows()):
    result = load_phasor(row)
    if result is None:
        print(f"[{i+1}/{n_total}] SKIP {row['filename']}")
        continue

    G_cal, S_cal = result["G_cal"], result["S_cal"]
    photons, mask = result["photons"], result["mask"]
    valid = mask & np.isfinite(G_cal) & np.isfinite(S_cal)

    if not valid.any():
        continue

    phasor_cache[row["filename"]] = result

    G_v  = G_cal[valid].ravel()
    S_v  = S_cal[valid].ravel()
    n_v  = photons[valid].ravel()
    wtot = float(n_v.sum())

    if wtot > 0:
        G_wm = float(np.dot(G_v, n_v) / wtot)
        S_wm = float(np.dot(S_v, n_v) / wtot)
    else:
        G_wm = float(np.nanmean(G_v))
        S_wm = float(np.nanmean(S_v))

    phi_rad    = np.arctan2(S_wm, G_wm)
    M          = np.sqrt(G_wm ** 2 + S_wm ** 2)
    tau_phi_ns = np.tan(phi_rad) / omega_ns if G_wm > 0 else np.nan
    tau_mod_ns = (np.sqrt(1.0 / M ** 2 - 1.0) / omega_ns
                  if 0.0 < M < 1.0 else np.nan)

    phasor_records.append({
        "filename":            row["filename"],
        "session_root":        row.get("session_root"),
        "fixation_type":       row.get("fixation_type"),
        "cell_type":           row.get("cell_type"),
        "em_filter_nm":        row.get("em_filter_nm"),
        "position_annotation": row.get("position_annotation"),
        "acquisition_time":    row.get("acquisition_time"),
        "n_px_total":          int(mask.size),
        "n_px_masked":         int(valid.sum()),
        "otsu_thresh":         float(result["thresh"]),
        "G_cal_mean":          float(np.nanmean(G_v)),
        "S_cal_mean":          float(np.nanmean(S_v)),
        "G_cal_wmean":         G_wm,
        "S_cal_wmean":         S_wm,
        "G_cal_std":           float(np.nanstd(G_v)),
        "S_cal_std":           float(np.nanstd(S_v)),
        "M_wmean":             float(M),
        "tau_phi_ns":          tau_phi_ns,
        "tau_mod_ns":          tau_mod_ns,
    })

    if (i + 1) % 20 == 0 or (i + 1) == n_total:
        print(f"  {i+1}/{n_total} done  ({len(phasor_records)} records so far)")

phasor_df = pd.DataFrame(phasor_records)
print(f"\nPhasor summary: {len(phasor_df)} files")
if not phasor_df.empty:
    print(phasor_df.groupby(["fixation_type", "em_filter_nm"])[
        ["G_cal_wmean", "S_cal_wmean", "tau_phi_ns", "tau_mod_ns"]
    ].mean().round(4).to_string())

phasor_df.to_csv(results_dir / "sdt_phasor_summary.csv", index=False)
print(f"\nSaved: {results_dir / 'sdt_phasor_summary.csv'}")

## Step 13: Phasor plots on universal semicircle

One figure per (fixation_type, em_filter_nm).
Rows = position annotation; columns = cell_type.
Hexbin weighted by photon count, drawn from the phasor_cache populated
in Step 12 (runs fast -- no SDT reloading needed).

In [ ]:
def _phasor_hexbin_ax(ax, filenames, phasor_cache,
                      gridsize: int = 50, cmap: str = "plasma") -> int:
    """
    Fill ax with a photon-weighted hexbin phasor plot using smoothed G/S.
    Returns the number of valid files plotted.
    """
    all_G, all_S, all_w = [], [], []
    n_plotted = 0
    for fn in filenames:
        result = phasor_cache.get(fn)
        if result is None:
            continue
        # Use smoothed phasor for display; fall back to raw if smoothing failed
        G_plot = result.get("G_sm", result["G_cal"])
        S_plot = result.get("S_sm", result["S_cal"])
        photons, mask = result["photons"], result["mask"]
        valid = mask & np.isfinite(G_plot) & np.isfinite(S_plot)
        if not valid.any():
            continue
        all_G.append(G_plot[valid].ravel())
        all_S.append(S_plot[valid].ravel())
        all_w.append(photons[valid].ravel())
        n_plotted += 1

    draw_semicircle(ax)
    if all_G:
        G_arr = np.concatenate(all_G)
        S_arr = np.concatenate(all_S)
        w_arr = np.concatenate(all_w)
        hb = ax.hexbin(G_arr, S_arr, C=w_arr,
                       reduce_C_function=np.sum,
                       gridsize=gridsize, cmap=cmap, mincnt=1)
        plt.colorbar(hb, ax=ax, label="photons", fraction=0.046, pad=0.04)

    ax.set_xlabel("G (calibrated)")
    ax.set_ylabel("S (calibrated)")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 0.55)
    ax.set_aspect("equal")
    return n_plotted


# Apply annotation remap and exclusions for plotting
plot_df = sample_df.copy()
plot_df["position_annotation"] = (
    plot_df["position_annotation"].replace(ANNOTATION_REMAP)
)
plot_df = plot_df[~plot_df["position_annotation"].isin(ANNOTATION_EXCLUDE)]

# Apply same remap to phasor summary so mean scatter uses consistent labels
plot_phasor_df = phasor_df.copy()
plot_phasor_df["position_annotation"] = (
    plot_phasor_df["position_annotation"].replace(ANNOTATION_REMAP)
)
plot_phasor_df = plot_phasor_df[
    ~plot_phasor_df["position_annotation"].isin(ANNOTATION_EXCLUDE)
]

_CT_COLORS = plt.cm.tab10.colors

for (fix_type, em_nm), grp in plot_df.groupby(
    ["fixation_type", "em_filter_nm"], dropna=False
):
    _present    = set(grp["cell_type"].dropna().unique())
    cell_types  = [ct for ct in CELL_TYPE_ORDER if ct in _present] + \
                  sorted(_present - set(CELL_TYPE_ORDER))
    annotations = sorted(grp["position_annotation"].unique())

    annot_rows = ["(all)"] + [a for a in annotations if a != "(all)"]
    n_rows = len(annot_rows)
    n_cols = max(len(cell_types), 1)

    # -- Hexbin grid (per-pixel, smoothed) -----------------------------------
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5.5 * n_cols, 5.0 * n_rows),
                             squeeze=False)

    for r_idx, annot in enumerate(annot_rows):
        sub = grp if annot == "(all)" else grp[grp["position_annotation"] == annot]

        for c_idx, ct in enumerate(cell_types if cell_types else [None]):
            ax = axes[r_idx][c_idx]
            cell_sub = sub[sub["cell_type"] == ct] if ct is not None else sub
            ct_label = ct if ct is not None else "all"
            fns      = cell_sub["filename"].tolist()
            n_plotted = _phasor_hexbin_ax(ax, fns, phasor_cache)
            ax.set_title(f"{ct_label}  n={n_plotted}\n{annot}", fontsize=8)

    fig.suptitle(f"Phasor (per-pixel, smoothed)  {fix_type} / {em_nm} nm",
                 fontsize=11)
    plt.tight_layout()
    plt.show()

    # -- Per-file mean phasor scatter ----------------------------------------
    ps_grp = plot_phasor_df[
        (plot_phasor_df["fixation_type"] == fix_type) &
        (plot_phasor_df["em_filter_nm"]  == em_nm)
    ]
    if ps_grp.empty:
        continue

    fig2, ax2 = plt.subplots(figsize=(6, 5))
    draw_semicircle(ax2)

    for i, ct in enumerate(cell_types if cell_types else [None]):
        ct_sub = ps_grp[ps_grp["cell_type"] == ct] if ct is not None else ps_grp
        color  = _CT_COLORS[i % len(_CT_COLORS)]
        for annot, ann_sub in ct_sub.groupby("position_annotation", dropna=False):
            ax2.scatter(
                ann_sub["G_cal_wmean"], ann_sub["S_cal_wmean"],
                color=color, label=f"{ct} / {annot}" if ct else str(annot),
                s=35, alpha=0.75, edgecolors="none",
            )

    ax2.set_xlabel("G_cal (per-file wmean)")
    ax2.set_ylabel("S_cal (per-file wmean)")
    ax2.set_xlim(-0.05, 1.05)
    ax2.set_ylim(-0.05, 0.55)
    ax2.set_aspect("equal")
    ax2.legend(fontsize=7, loc="upper left")
    ax2.set_title(
        f"Per-file mean phasor  {fix_type} / {em_nm} nm  "
        f"(n={len(ps_grp)} files)",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()

## Step 13b: Per-file phasor drill-down

Plots one hexbin panel per file for a configurable (annotation, fixation, channel,
cell_types) slice.  Useful for checking whether a group-level pattern is driven
by one or two outlier files.  The white cross marks the intensity-weighted mean
for that file.

In [ ]:
# -- Filter configuration -- adjust as needed ---------------------------------
DRILL_ANNOTATION = "colony_deep"   # raw annotation value (pre-remap)
DRILL_FIXATION   = "form"
DRILL_EM_NM      = 457
DRILL_CELL_TYPES = None            # None = all; or e.g. ["BKO", "KPCWT"]

# Resolve through ANNOTATION_REMAP so the label matches stored values
_drill_annot = ANNOTATION_REMAP.get(DRILL_ANNOTATION, DRILL_ANNOTATION)

drill_df = sample_df.copy()
drill_df["_annot_mapped"] = drill_df["position_annotation"].map(
    lambda a: ANNOTATION_REMAP.get(str(a), str(a))
)
drill_df = drill_df[
    (drill_df["_annot_mapped"] == _drill_annot) &
    (drill_df["fixation_type"] == DRILL_FIXATION) &
    (drill_df["em_filter_nm"]  == DRILL_EM_NM) &
    drill_df["filename"].isin(phasor_cache)
]
if DRILL_CELL_TYPES is not None:
    drill_df = drill_df[drill_df["cell_type"].isin(DRILL_CELL_TYPES)]
drill_df = drill_df.sort_values(["cell_type", "acquisition_time"]).reset_index(drop=True)

print(f"Files matching filter ({_drill_annot} / {DRILL_FIXATION} / {DRILL_EM_NM} nm):"
      f"  {len(drill_df)}")
if drill_df.empty:
    print("  No files -- check DRILL_ANNOTATION / DRILL_FIXATION / DRILL_EM_NM.")
else:
    print(drill_df[["filename", "cell_type", "session_root"]].to_string(index=False))

    # Per-file means for the cross marker
    means_idx = plot_phasor_df.set_index("filename") if "filename" in plot_phasor_df.columns \
        else pd.DataFrame()

    ncols = min(4, len(drill_df))
    nrows = (len(drill_df) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.5 * ncols, 4.0 * nrows),
                             squeeze=False)

    for idx, (_, row) in enumerate(drill_df.iterrows()):
        r, c  = divmod(idx, ncols)
        ax    = axes[r][c]
        fn    = row["filename"]
        res   = phasor_cache[fn]
        mask  = res["mask"]
        G_sm  = res.get("G_sm", res["G_cal"])
        S_sm  = res.get("S_sm", res["S_cal"])
        ph    = res["photons"]
        valid = mask & np.isfinite(G_sm) & np.isfinite(S_sm)

        draw_semicircle(ax)
        if valid.any():
            hb = ax.hexbin(
                G_sm[valid].ravel(), S_sm[valid].ravel(),
                C=ph[valid].ravel(), reduce_C_function=np.sum,
                gridsize=40, cmap="plasma", mincnt=1,
            )
            plt.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)

        # Cross at the per-file intensity-weighted mean
        if fn in means_idx.index:
            mx = means_idx.loc[fn, "G_cal_wmean"]
            my = means_idx.loc[fn, "S_cal_wmean"]
            ax.plot(mx, my, "w+", ms=10, mew=1.8, zorder=5)

        ct    = str(row.get("cell_type", ""))
        sess  = str(row.get("session_root", ""))[:8]
        stem  = fn.rsplit("_", 1)[-1] if "_" in fn else fn
        ax.set_title(f"{ct}  {sess}\n{stem}", fontsize=7)
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 0.55)
        ax.set_xlabel("G (cal)", fontsize=7)
        ax.set_ylabel("S (cal)", fontsize=7)
        ax.tick_params(labelsize=6)

    for idx in range(len(drill_df), nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r][c].set_visible(False)

    fig.suptitle(
        f"Per-file phasor: {_drill_annot} / {DRILL_FIXATION} / {DRILL_EM_NM} nm"
        f"  (n={len(drill_df)} files, white cross = file mean)",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

## Summary

Phase D complete.

Computed:
- Per-pixel calibrated phasor (G_cal, S_cal) from raw SDT decays
- Otsu intensity mask per image (top fraction of pixels by photon count)
- Photon-count weighted mean phasor per file
- Phase lifetime (tau_phi) and modulation lifetime (tau_mod)
- Hexbin phasor plots on the universal semicircle

Saved:
  results/sdt_phasor_summary.csv  (one row per sample file)

phasor_cache: dict {filename -> {G_cal, S_cal, photons, mask, thresh}}
Available in-session for further analysis.

Before Phase E (fit analysis):
  1. Verify tau_phi_ns and tau_mod_ns are in expected NADH range (~0.5-3 ns)
  2. Check that phasor cloud sits on or inside the universal semicircle
  3. If a2/tau2/a3/tau3 have been re-exported from SPCImage, re-run Phase A -> C

In [ ]:
import winsound as _ws, time as _t
for _ in range(3):
    _ws.Beep(1000, 400)
    _t.sleep(1)